# B4: When and for whom do decision features help?

Four observational views and one prediction rule: **datasets** (label-free properties, including decision redundancy — how well the non-text columns predict a model's joint decision output — with one Benjamini-Hochberg adjustment over all properties and predictors, no primary property), **decision models** (Group A diagnostics against usefulness; exploratory, using the selected cohort), **text columns** (refits without one column's decision block, all seven predictors; no permutation importance, shuffling or SHAP), **rows** (availability and length, baseline uncertainty, decision confidence, cross-model agreement) and a **leave-one-dataset-out rule** that predicts each dataset's gain in advance from three nested label-free input sets.

Seven predictors use prespecified settings on the same five outer folds across 18 datasets (3 binary, 6 multiclass, 9 regression). They are linear/logistic regression, CatBoost, LightGBM, TabPFN-3.5, TabICL v2, LimiX-2 and Kumo. Each structured baseline includes the non-text columns and one text-availability flag per text column. Fixed headline metrics: binary ROC-AUC, multiclass macro one-vs-rest ROC-AUC, regression R². Datasets are the unit: each contributes its five-fold mean paired gain; raw gains stay within a task family (ΔAUC and ΔR², N = 9 each, separate panels) and only the headroom share (gain / (1 − baseline)) pools all 18 datasets. Means sit next to medians; intervals resample datasets (2,000 draws). Decision medians are taken over all members of the recorded decision-model cohort; nothing selects a winner.

Reads `output_JEVPFN/B/B4/results/group_b/b4_when_text_helps.<cohort>-<roster digest>.zip`; label-free field types (section 4) are computed from the local datasets.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import pandas as pd
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import benchmark_export, group_b, group_b_views as views, learner_scores, study_export
from src.visualize import critical_difference, figures, style
from src.visualize import group_b as gbfig
from src.visualize import when_text_helps as b3fig
style.apply()
NOTEBOOK = "B/B4"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. What is complete: export, provenance and coverage")
display_json("Notebook execution identity", provenance())
frames, receipt = benchmark_export.read("B4")
EXPORTS = {"B4": (frames, receipt)}
ROSTER = benchmark_export.decision_roster()
display_json("Decision-model cohort and members", benchmark_export.cohort_record())
WAITING = "Waiting for export: no measurements are shown."
for study, (frames_, receipt_) in EXPORTS.items():
    print(f"{study}: {study_export.describe(receipt_)}")
    record({k: receipt_.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, f"{study} export identity")
    if frames_:
        display_table(f"{study}: planned versus complete five-fold arms per phase and predictor", frames_["coverage"], preview_rows=14)
        if "median_roster_gaps" in frames_ and not frames_["median_roster_gaps"].empty:
            display_table(f"{study}: dataset x predictor cells whose median lacks configured decision models", frames_["median_roster_gaps"])

B4: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\B\B4\results\group_b\b4_when_text_helps.open_weight-5baeccd37d70d034.zip is absent. No measurements are shown.


## 2. Summary: datasets

Top: the median model's gain against the baseline headroom, per task family, with the ceiling diagonal. Bottom: Spearman ρ of every label-free property with that gain, per predictor (asterisk: BH q ≤ 0.10).

In [2]:
report_section("2. Datasets: headroom and label-free properties")
if not frames:
    print(WAITING)
else:
    gains = frames["dataset_gains"]
    medians = group_b.median_over_decision_models(gains, roster=ROSTER)
    headroom = views.headroom_table(medians)
    gbfig.render(save, b3fig.headroom_and_associations(headroom, frames["associations"], name="b4_headroom_and_properties", what="When decision features help: datasets"))
    display_table("Label-free dataset properties", frames["properties"], preview_rows=18)
    display_table("Decision redundancy per dataset and model (out-of-fold R² from non-text columns)", frames["redundancy"], preview_rows=12)
    display_table("Spearman associations with BH q-values (all properties, all predictors)", frames["associations"], preview_rows=12)

Waiting for export: no measurements are shown.


## 3. Decision models: diagnostics and usefulness

This exploratory analysis uses the recorded cohort; size, data and readout are confounded. Usefulness is the equal-dataset mean headroom share (all 18 datasets) of each model's both view, averaged over predictors; ρ and BH q are recomputed on that outcome (the export's `model_relations` table, on raw gain, is listed for completeness).

In [3]:
report_section("3. Decision models: diagnostics and usefulness")
if not frames:
    print(WAITING)
else:
    usefulness, relations = views.model_usefulness_relations(gains, frames["model_diagnostics"], roster=ROSTER)
    gbfig.render(save, b3fig.diagnostics_vs_usefulness(usefulness, relations, name="b4_model_diagnostics_vs_usefulness", what="When decision features help: decision models"))
    display_table("Per decision model: usefulness and Group A diagnostics", usefulness, preview_rows=17)
    display_table("Spearman relations across decision models (headroom share; BH over the diagnostics)", relations)
    display_table("Export model_relations (raw gain within task family; headroom share over all datasets)", frames["model_relations"], preview_rows=12)

Waiting for export: no measurements are shown.


## 4. Text columns: refits without one column

In [4]:
report_section("4. Text columns: refits without one column")
if not frames or frames["column_importance"].empty:
    print(WAITING if not frames else "Text-column refits are pending in this export.")
else:
    from src.evaluation.field_types import field_type_table
    columns = views.column_refits(frames["column_importance"], field_types=field_type_table())
    gbfig.render(save, b3fig.column_refits(columns, name="b4_text_column_refits", what="When decision features help: text columns"))
    display_table("Per text column: median over models and predictors of the drop without its block", columns, preview_rows=20)

Waiting for export: no measurements are shown.


## 5. Rows: who benefits?

Row benefit = no-text per-row loss minus the arm's (log loss; regression squared error over the training-target variance), within fixed label-free cohorts, then equal dataset weight.

In [5]:
report_section("5. Rows: who benefits?")
if not frames or frames["row_cohorts"].empty:
    print(WAITING if not frames else "Row cohorts are pending in this export.")
else:
    cohorts = views.cohort_benefits(frames["row_cohorts"])
    gbfig.render(save, b3fig.cohort_benefits(cohorts, name="b4_row_cohorts", what="When decision features help: rows"))
    display_table("Equal-dataset mean benefit per cohort, view and task family (mean over predictors)", cohorts, preview_rows=14)

Waiting for export: no measurements are shown.


## 6. Predicting the gain in advance

Ridge (α = 1) on standardized label-free properties, fitted without the held-out dataset: pre-extraction (text length), after a pilot of up to 200 rows per decision model (+ pilot redundancy) and post-extraction (+ redundancy on all rows). Reported: rank correlation, use/don't-use accuracy, regret against always and never using text, and a 2,000-shuffle null.

In [6]:
report_section("6. Predicting the gain in advance")
if not frames or frames["lodo"].empty:
    print(WAITING if not frames else "Leave-one-dataset-out results are pending in this export.")
else:
    gbfig.render(save, b3fig.lodo(frames["lodo_predictions"], frames["lodo"], name="b4_lodo_rule", what="Predicting the gain in advance"))
    display_table("Leave-one-dataset-out evaluation per predictor and input set", frames["lodo"], preview_rows=21)

Waiting for export: no measurements are shown.


## 7. Statistics

The exploratory tests compare the regret of the three routing rules and the always-use and never-use policies. Holm adjustment and Bayesian signed-rank probabilities are reported within each predictor.

In [7]:
report_section("7. Statistics")
framework = learner_scores.run_application("B4_when_text_helps")
print(f"B4_when_text_helps: {framework['status']} (framework.status = {framework['framework_status']})")
display_json("Framework application B4_when_text_helps", learner_scores.summary_record(framework))
gbfig.render(save, critical_difference.cd_figures(framework))

B4_when_text_helps: awaiting_export (framework.status = approved)


In [8]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified exports and local caches named above were read; no inference, model training or VSC changes were performed.")

B/B4

1. What is complete: export, provenance and coverage


Notebook execution identity
{
  "code_sha256": "253d5f17bdea4be35b0e853055326f9a06bd5d0a5b2b0c789f83d236fd3c5074",
  "git_head": "0929454751c9921410a46b29506eb6c2b4ca280f",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


Decision-model cohort and members
{
  "decision_cohort": "open_weight",
  "decision_roster": [
    "liquid",
    "liquid_lfm3",
    "perplexity",
    "cloudflare",
    "strands",
    "intern",
    "h2o",
    "sii",
    "kev",
    "decision3",
    "nimble",
    "glide",
    "mammoth",
    "drex_max",
    "knowline",
  